# 🤖 Generative AI Interview Questions — Theory + Practical Code
> Covers: Generative Models, LLMs, Multimodal Models, Embeddings, Training & Evaluation

---
Each section has:
- **Theory Summary** — concise explanation
- **Code Demo** — runnable Python to reinforce the concept

> ⚠️ Some cells use `torch`, `numpy`, `sklearn`. Install via: `pip install torch numpy scikit-learn`

In [7]:
# ============================================================
# GLOBAL IMPORTS — run this cell first
# ============================================================
import numpy as np
import math
import random
from collections import Counter

try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    TORCH_AVAILABLE = True
    print(f'PyTorch {torch.__version__} ready ✅')
except ImportError:
    TORCH_AVAILABLE = False
    print('PyTorch not installed. Install with: pip install torch')

print('NumPy', np.__version__, 'ready ✅')

PyTorch 2.10.0+cpu ready ✅
NumPy 1.26.4 ready ✅


---
# 🔷 SECTION 1: Generative Models
---

## Q1. Generative vs Discriminative Models

| | Generative | Discriminative |
|---|---|---|
| **Goal** | Learn P(X) or P(X,Y) | Learn P(Y|X) |
| **Examples** | VAE, GAN, GPT | SVM, Logistic Regression, BERT |
| **Use case** | Create new data | Classify/label data |

**Generative** models learn the underlying data distribution and can *generate* new samples.(An artist who can paint new art from scratch)  
**Discriminative** models learn decision boundaries between classes.(A judge who can tell real art from fake art)

In [8]:
# Q1 CODE: Naive Bayes (Generative) vs Logistic Regression (Discriminative)
import numpy as np

# --- Generative Model: Naive Bayes (models P(X|Y) and P(Y)) ---
class NaiveBayesGenerative:
    def fit(self, X, y):
        self.classes = np.unique(y)
        self.priors = {}
        self.likelihoods = {}   # mean and std per feature per class
        for c in self.classes:
            X_c = X[y == c]
            self.priors[c] = len(X_c) / len(X)
            self.likelihoods[c] = {'mean': X_c.mean(0), 'std': X_c.std(0) + 1e-9}

    def predict(self, X):
        preds = []
        for x in X:
            posteriors = {}
            for c in self.classes:
                log_prior = np.log(self.priors[c])
                m, s = self.likelihoods[c]['mean'], self.likelihoods[c]['std']
                log_likelihood = np.sum(-0.5 * ((x - m) / s)**2 - np.log(s))
                posteriors[c] = log_prior + log_likelihood
            preds.append(max(posteriors, key=posteriors.get))
        return np.array(preds)

# Toy data: 2 classes, 2 features
np.random.seed(42)
X = np.vstack([np.random.randn(50, 2) + [2,2],   # class 0
               np.random.randn(50, 2) + [-2,-2]]) # class 1
y = np.array([0]*50 + [1]*50)

model = NaiveBayesGenerative()
model.fit(X, y)
preds = model.predict(X)
acc = (preds == y).mean()
print(f'Naive Bayes (Generative) Accuracy: {acc:.2f}')
print('Key point: Naive Bayes models the joint P(X,Y), allowing us to GENERATE X given Y')
print('Example generation from class 0:')
c = 0
sample = np.random.normal(model.likelihoods[c]['mean'], model.likelihoods[c]['std'])
print(f'  Generated sample: {sample.round(2)}')

Naive Bayes (Generative) Accuracy: 1.00
Key point: Naive Bayes models the joint P(X,Y), allowing us to GENERATE X given Y
Example generation from class 0:
  Generated sample: [2.16 2.47]


## Q2. GAN Architecture — Generator & Discriminator

- **Generator (G)**: Takes random noise z → produces fake data
- **Discriminator (D)**: Takes real/fake data → outputs probability of being real
- **Training**: Minimax game — G tries to fool D; D tries to distinguish real from fake
- **Loss**: `min_G max_D [log D(x) + log(1 - D(G(z)))]`

In [ ]:
# Q2 CODE: Minimal GAN training loop (1D data)
import numpy as np

def sigmoid(x): return 1 / (1 + np.exp(-np.clip(x, -500, 500)))

# Real data: Gaussian N(4, 0.5)
def real_data(n): return np.random.normal(4, 0.5, (n, 1))

# Generator: noise → scalar (using learned mean and std)
g_params = {'mean': np.array([0.0]), 'std': np.array([1.0])}

def generate(n):
    z = np.random.randn(n, 1)
    return g_params['mean'] + np.abs(g_params['std']) * z

# Discriminator: linear with sigmoid output
d_w, d_b = np.array([[1.0]]), np.array([0.0])

def discriminate(x):
    return sigmoid(x @ d_w + d_b)

lr = 0.01
print('GAN Training (simplified 1D demo)')
print(f'Target distribution: N(4, 0.5)')
print(f'Generator starts at: mean={g_params["mean"][0]:.2f}, std={g_params["std"][0]:.2f}')

for step in range(500):
    # Train D
    real = real_data(32)
    fake = generate(32)
    d_real_loss = -np.mean(np.log(discriminate(real) + 1e-8))
    d_fake_loss = -np.mean(np.log(1 - discriminate(fake) + 1e-8))
    # Gradient update for D (simplified)
    d_w += lr * (np.mean(real) - np.mean(fake)) * 0.01

    # Train G: push G output toward real
    g_params['mean'] += lr * (4.0 - g_params['mean'][0]) * 0.1
    g_params['std']  += lr * (0.5 - np.abs(g_params['std'][0])) * 0.1

print(f'Generator after training: mean={g_params["mean"][0]:.2f}, std={g_params["std"][0]:.2f}')
print(f'Real data mean: ~4.0, std: ~0.5')
print('\nKey insight: Generator learns to mimic real distribution through competition with Discriminator')

## Q3. Variational Autoencoder (VAE)

- **Encoder**: X → μ (mean) and σ (std) of latent distribution
- **Reparameterization**: z = μ + ε·σ  (ε ~ N(0,1)) — makes it differentiable
- **Decoder**: z → reconstructed X
- **Loss**: Reconstruction loss + KL divergence (regularizes latent space to be Gaussian)

In [ ]:
# Q3 CODE: VAE core concepts
import numpy as np

# --- The Reparameterization Trick (most important VAE concept) ---
def reparameterize(mu, log_var):
    """Sample z = mu + eps * std  where eps ~ N(0,1)"""
    std = np.exp(0.5 * log_var)
    eps = np.random.randn(*mu.shape)
    return mu + eps * std  # differentiable w.r.t. mu and log_var!

# --- KL Divergence: D_KL(N(mu,sigma) || N(0,1)) ---
def kl_divergence(mu, log_var):
    """Closed-form KL divergence for Gaussian"""
    return -0.5 * np.sum(1 + log_var - mu**2 - np.exp(log_var))

# --- Reconstruction Loss (MSE) ---
def reconstruction_loss(x_orig, x_recon):
    return np.mean((x_orig - x_recon)**2)

# Demo: Encode a point, sample from latent space, compute VAE loss
np.random.seed(42)

# Simulate encoder output for a batch of 5 samples, latent_dim=2
mu      = np.array([[0.5, -0.3], [1.2, 0.4], [-0.5, 0.8], [0.0, 0.0], [2.0, -1.0]])
log_var = np.array([[-0.5, -0.3], [-0.2, -0.5], [-0.8, -0.2], [0.0, 0.0], [-0.4, -0.6]])

# Reparameterize
z = reparameterize(mu, log_var)
print('Encoder output (mu):', mu[0])
print('Sampled z:          ', z[0].round(3))

# Simulate decoder: just reconstruct from z (simplified)
x_original    = np.random.randn(5, 10)
x_reconstructed = x_original + np.random.randn(5, 10) * 0.1  # near-perfect reconstruction

recon_loss = reconstruction_loss(x_original, x_reconstructed)
kl_loss    = kl_divergence(mu, log_var)
beta = 1.0  # weight for KL term
total_vae_loss = recon_loss + beta * kl_loss

print(f'\nReconstruction Loss: {recon_loss:.4f}')
print(f'KL Divergence:       {kl_loss:.4f}')
print(f'Total VAE Loss:      {total_vae_loss:.4f}')
print('\nKL pushes latent space toward N(0,1), enabling GENERATION by sampling z ~ N(0,1)')

## Q4. Conditional vs Unconditional Generative Models

- **Unconditional**: Generate samples from P(X) — no control over output
- **Conditional**: Generate from P(X|Y) — control output class/attributes
- **Example**: Conditional GAN generates dog images of a *specific breed*

In [ ]:
# Q4 CODE: Conditional vs Unconditional generation demo
import numpy as np

# Unconditional generator: samples from mixture of Gaussians (no control)
def unconditional_generate(n=5):
    samples = []
    for _ in range(n):
        # Randomly picks any class — no control!
        class_id = np.random.randint(0, 3)
        means = {0: [2,2], 1: [-2,-2], 2: [2,-2]}
        sample = np.random.normal(means[class_id], 0.3)
        samples.append((class_id, sample))
    return samples

# Conditional generator: given label y, generate X ~ P(X|Y=y)
def conditional_generate(y, n=5):
    class_means = {0: [2,2], 1: [-2,-2], 2: [2,-2]}
    class_names = {0: 'Poodle', 1: 'Labrador', 2: 'Husky'}
    samples = []
    for _ in range(n):
        sample = np.random.normal(class_means[y], 0.3)
        samples.append(sample)
    return samples, class_names[y]

print('=== Unconditional Generation ===' )
for cls, samp in unconditional_generate(5):
    print(f'  Random class {cls}: {samp.round(2)}')

print('\n=== Conditional Generation (requesting class=1: Labrador) ===')
samples, name = conditional_generate(y=1, n=5)
for s in samples:
    print(f'  {name}: {s.round(2)}')

print('\n✅ Conditional models give precise control over what gets generated!')

## Q5. Mode Collapse in GANs

**Mode Collapse**: Generator produces only a few types of outputs, ignoring data diversity.

**Solutions**:
1. Mini-batch discrimination
2. Diverse training data
3. Wasserstein GAN (WGAN)
4. Ensemble methods
5. Architectural changes (PGGANs)

In [ ]:
# Q5 CODE: Detecting mode collapse via output diversity
import numpy as np

def diversity_score(samples, bins=10, feature_range=(-4, 4)):
    """Measures how spread generated samples are (higher = more diverse)"""
    hist, _ = np.histogram(samples, bins=bins, range=feature_range, density=True)
    hist = hist + 1e-10
    entropy = -np.sum(hist * np.log(hist))
    return entropy

np.random.seed(42)

# Scenario 1: MODE COLLAPSE — generator only outputs ~1 value
collapsed_samples = np.random.normal(loc=2.0, scale=0.05, size=1000)

# Scenario 2: HEALTHY generator — samples the full distribution
# Real data is a mixture of 3 Gaussians
healthy_samples = np.concatenate([
    np.random.normal(-2, 0.3, 333),
    np.random.normal( 0, 0.3, 334),
    np.random.normal( 2, 0.3, 333)
])

# Mini-batch discrimination: penalize low variance within a batch
def minibatch_diversity(batch):
    """Returns diversity statistic for a mini-batch"""
    diffs = []
    for i in range(len(batch)):
        for j in range(i+1, len(batch)):
            diffs.append(abs(batch[i] - batch[j]))
    return np.mean(diffs)

collapsed_batch = collapsed_samples[:16]
healthy_batch   = healthy_samples[:16]

print(f'Mode Collapsed Generator:')
print(f'  All outputs near: {collapsed_samples.mean():.2f} ± {collapsed_samples.std():.4f}')
print(f'  Diversity score (entropy): {diversity_score(collapsed_samples):.3f}')
print(f'  Mini-batch diversity: {minibatch_diversity(collapsed_batch):.4f}')

print(f'\nHealthy Generator:')
print(f'  Outputs spread: mean={healthy_samples.mean():.2f} ± std={healthy_samples.std():.2f}')
print(f'  Diversity score (entropy): {diversity_score(healthy_samples):.3f}')
print(f'  Mini-batch diversity: {minibatch_diversity(healthy_batch):.4f}')
print('\n✅ High entropy + high batch diversity = no mode collapse')

## Q6. Overfitting in Generative Models

**Signs**: Memorizes training data; poor diversity; no generalization.

**Prevention techniques**:
- Dropout, Weight Decay, Batch Normalization
- Early Stopping
- Data Augmentation
- Cross-Validation

In [ ]:
# Q6 CODE: Regularization techniques demo
import numpy as np

# --- L2 Weight Decay ---
def l2_regularized_loss(loss, weights, lambda_=0.01):
    """Penalizes large weights to prevent memorization"""
    l2_penalty = lambda_ * np.sum([np.sum(w**2) for w in weights])
    return loss + l2_penalty

# --- Dropout (training-time noise injection) ---
def dropout(activations, drop_rate=0.5, training=True):
    """Randomly zeros activations during training"""
    if not training:
        return activations  # no dropout at inference!
    mask = np.random.rand(*activations.shape) > drop_rate
    return activations * mask / (1 - drop_rate)  # scale to maintain expectation

# --- Early Stopping ---
class EarlyStopping:
    def __init__(self, patience=5):
        self.patience = patience
        self.best_loss = float('inf')
        self.counter = 0

    def step(self, val_loss):
        if val_loss < self.best_loss:
            self.best_loss = val_loss
            self.counter = 0
            return False  # continue training
        self.counter += 1
        if self.counter >= self.patience:
            return True   # stop training!
        return False

# Demo
np.random.seed(42)
activations = np.array([0.8, 1.2, -0.5, 2.1, 0.3, -1.5])
print(f'Original activations: {activations}')
print(f'After dropout (50%):  {dropout(activations, 0.5).round(3)}')
print(f'At inference (no dropout): {dropout(activations, 0.5, training=False)}')

weights = [np.random.randn(10,10), np.random.randn(10,1)]
base_loss = 0.5
reg_loss = l2_regularized_loss(base_loss, weights, lambda_=0.01)
print(f'\nBase loss: {base_loss:.4f}, L2-regularized loss: {reg_loss:.4f}')

print('\nEarly Stopping demo (simulated val losses):')
es = EarlyStopping(patience=3)
val_losses = [1.0, 0.8, 0.7, 0.75, 0.8, 0.85]
for epoch, vl in enumerate(val_losses):
    stop = es.step(vl)
    print(f'  Epoch {epoch+1}: val_loss={vl:.2f} | Stop={stop}')
    if stop:
        print('  ⛔ Early stopping triggered!')
        break

## Q7. Gradient Clipping

**Problem**: Exploding gradients destabilize training (especially in GANs/RNNs).

**Gradient Clipping**: If ‖∇‖ > threshold, scale gradient so ‖∇‖ = threshold.

This prevents:
- Numerically unstable weight updates
- Oscillating loss curves

In [ ]:
# Q7 CODE: Gradient clipping implementation
import numpy as np

def clip_gradients_by_norm(gradients, max_norm=1.0):
    """Clips gradient list so global norm <= max_norm"""
    total_norm = np.sqrt(sum(np.sum(g**2) for g in gradients))
    clip_coef = max_norm / (total_norm + 1e-8)
    if clip_coef < 1.0:
        gradients = [g * clip_coef for g in gradients]
    return gradients, total_norm, min(total_norm, max_norm)

def clip_gradients_by_value(gradients, clip_value=1.0):
    """Clips each gradient element to [-clip_value, clip_value]"""
    return [np.clip(g, -clip_value, clip_value) for g in gradients]

# Simulate gradients: normal gradients vs exploding gradients
np.random.seed(42)

normal_grads   = [np.random.randn(10) * 0.1 for _ in range(3)]
exploding_grads = [np.random.randn(10) * 100 for _ in range(3)]

# Norm clipping
norm_before_normal = np.sqrt(sum(np.sum(g**2) for g in normal_grads))
norm_before_explod = np.sqrt(sum(np.sum(g**2) for g in exploding_grads))

clipped_normal, bn, an = clip_gradients_by_norm(normal_grads, max_norm=1.0)
clipped_explod, be, ae = clip_gradients_by_norm(exploding_grads, max_norm=1.0)

print('=== Gradient Clipping by Norm (max_norm=1.0) ===')
print(f'Normal gradient  — before: {bn:.4f}, after: {an:.4f}')
print(f'Exploding gradient — before: {be:.2f}, after: {ae:.4f}  ← clipped!')

print('\n=== Gradient Clipping by Value (clip_value=0.5) ===')
sample_grad = np.array([3.5, -2.1, 0.3, -0.8, 5.0])
clipped_val = clip_gradients_by_value([sample_grad], 0.5)[0]
print(f'Before: {sample_grad}')
print(f'After:  {clipped_val}')
print('\n✅ Clipping prevents parameter updates from being catastrophically large')

## Q8. Training with Limited Data

Strategies:
1. **Data Augmentation** — rotations, flips, noise
2. **Transfer Learning** — start from pre-trained weights
3. **Semi-supervised Learning** — use unlabeled data
4. **Progressive Training** (PGGANs)
5. **Data Synthesis** — use GANs to generate more data

In [ ]:
# Q8 CODE: Data augmentation strategies
import numpy as np

# Simulate a small dataset (10 samples × 5 features)
np.random.seed(42)
small_dataset = np.random.randn(10, 5)

def augment_with_noise(X, noise_level=0.05, n_augments=3):
    """Add Gaussian noise to create new samples"""
    augmented = [X]
    for _ in range(n_augments):
        noise = np.random.randn(*X.shape) * noise_level
        augmented.append(X + noise)
    return np.vstack(augmented)

def augment_with_mixup(X, y, alpha=0.2, n_augments=5):
    """Mixup: interpolate between pairs of samples"""
    augmented_X, augmented_y = [X], [y]
    for _ in range(n_augments):
        idx1 = np.random.randint(0, len(X))
        idx2 = np.random.randint(0, len(X))
        lam = np.random.beta(alpha, alpha)
        mixed_x = lam * X[idx1] + (1 - lam) * X[idx2]
        mixed_y = lam * y[idx1] + (1 - lam) * y[idx2]
        augmented_X.append(mixed_x.reshape(1,-1))
        augmented_y.append(mixed_y)
    return np.vstack(augmented_X), np.array(augmented_y)

y_small = np.random.randint(0, 2, 10).astype(float)

aug_noise  = augment_with_noise(small_dataset, noise_level=0.05, n_augments=4)
aug_mix_X, aug_mix_y = augment_with_mixup(small_dataset, y_small, n_augments=10)

print(f'Original dataset:          {small_dataset.shape[0]} samples')
print(f'After noise augmentation:  {aug_noise.shape[0]} samples (4x increase)')
print(f'After mixup augmentation:  {aug_mix_X.shape[0]} samples')
print(f'\nMixup sample 11 (blend of two real samples):')
print(f'  Features: {aug_mix_X[10].round(3)}')
print(f'  Label: {aug_mix_y[10]:.3f} (soft label between 0 and 1)')
print('\n✅ These strategies help prevent overfitting with limited data')

## Q9. Curriculum Learning

**Idea**: Train models on **easy examples first**, then gradually introduce **harder examples** — mimicking how humans learn.

**Benefits**:
- Faster convergence
- Better final performance
- Reduced overfitting

In [ ]:
# Q9 CODE: Curriculum Learning implementation
import numpy as np

# Simulate a dataset with varying difficulty
np.random.seed(42)

def create_dataset_with_difficulty(n=100):
    """Creates samples with a 'difficulty' score"""
    data = []
    for i in range(n):
        # Sentence length as proxy for difficulty
        length = np.random.randint(3, 50)
        vocab_rarity = np.random.uniform(0, 1)  # 0=common, 1=rare
        difficulty = 0.5 * length/50 + 0.5 * vocab_rarity
        data.append({'id': i, 'length': length, 'vocab_rarity': vocab_rarity,
                     'difficulty': difficulty})
    return data

class CurriculumScheduler:
    def __init__(self, data, n_stages=5):
        # Sort data by difficulty
        self.sorted_data = sorted(data, key=lambda x: x['difficulty'])
        self.n_stages = n_stages
        self.stage_size = len(data) // n_stages

    def get_stage_data(self, stage):
        """Returns data up to the current stage (cumulative)"""
        end_idx = min((stage + 1) * self.stage_size, len(self.sorted_data))
        return self.sorted_data[:end_idx]

dataset = create_dataset_with_difficulty(100)
scheduler = CurriculumScheduler(dataset, n_stages=5)

print('Curriculum Learning Schedule:')
for stage in range(5):
    stage_data = scheduler.get_stage_data(stage)
    difficulties = [d['difficulty'] for d in stage_data]
    print(f'  Stage {stage+1}: {len(stage_data)} samples | '
          f'difficulty range: {min(difficulties):.2f} – {max(difficulties):.2f}')

print('\n✅ Model sees easy examples first, complexity grows gradually across stages')

## Q10. Learning Rate Scheduling

**Why needed**: Fixed LR is too high late in training (overshoots) and too low early on (slow).

**Common strategies**:
- Step decay: reduce by factor every N epochs
- Exponential decay
- Cosine annealing
- Warmup → decay

In [ ]:
# Q10 CODE: Learning Rate Schedulers
import numpy as np

def step_decay(initial_lr, epoch, drop=0.5, epochs_drop=10):
    """Reduce LR by 'drop' every 'epochs_drop' epochs"""
    return initial_lr * (drop ** (epoch // epochs_drop))

def exponential_decay(initial_lr, epoch, decay_rate=0.95):
    return initial_lr * (decay_rate ** epoch)

def cosine_annealing(initial_lr, epoch, total_epochs, eta_min=1e-6):
    return eta_min + 0.5 * (initial_lr - eta_min) * (1 + np.cos(np.pi * epoch / total_epochs))

def warmup_cosine(initial_lr, epoch, warmup_epochs=5, total_epochs=100):
    if epoch < warmup_epochs:
        return initial_lr * (epoch + 1) / warmup_epochs  # linear warmup
    return cosine_annealing(initial_lr, epoch - warmup_epochs,
                            total_epochs - warmup_epochs)

epochs = list(range(0, 50, 5))
initial_lr = 0.1

print(f'{'Epoch':>6} | {'Step Decay':>12} | {'Exp Decay':>12} | {'Cosine':>12} | {'Warmup+Cos':>12}')
print('-' * 62)
for e in epochs:
    sd  = step_decay(initial_lr, e)
    ed  = exponential_decay(initial_lr, e)
    ca  = cosine_annealing(initial_lr, e, 50)
    wc  = warmup_cosine(initial_lr, e, warmup_epochs=5, total_epochs=50)
    print(f'{e:>6} | {sd:>12.6f} | {ed:>12.6f} | {ca:>12.6f} | {wc:>12.6f}')

## Q11. L1 vs L2 Loss

| | L1 (MAE) | L2 (MSE) |
|---|---|---|
| Formula | Σ|y - ŷ| | Σ(y - ŷ)² |
| Outliers | Robust | Sensitive |
| Output | Sharp edges | Smooth |
| Use case | Super-resolution, sparse outputs | Smooth textures |
| Gradient | Constant ±1 | Proportional to error |

In [ ]:
# Q11 CODE: L1 vs L2 loss comparison
import numpy as np

def l1_loss(y_true, y_pred):
    return np.mean(np.abs(y_true - y_pred))

def l2_loss(y_true, y_pred):
    return np.mean((y_true - y_pred)**2)

def l1_gradient(y_true, y_pred):
    return np.sign(y_pred - y_true)  # constant magnitude

def l2_gradient(y_true, y_pred):
    return 2 * (y_pred - y_true)     # proportional to error

# Scenario: most predictions are close, but one is a big outlier
y_true = np.array([2.0, 2.0, 2.0, 2.0, 2.0])
y_pred_good = np.array([2.1, 1.9, 2.05, 2.0, 100.0])  # one large outlier!

print('=== Impact of Outlier on L1 vs L2 ===')
print(f'Predictions: {y_pred_good}')
print(f'Ground truth: {y_true}')
print(f'L1 Loss: {l1_loss(y_true, y_pred_good):.4f}  ← less affected by outlier')
print(f'L2 Loss: {l2_loss(y_true, y_pred_good):.4f}  ← dominated by outlier!')

print('\n=== Gradient Behavior ===')
errors = np.array([0.01, 0.1, 0.5, 1.0, 5.0, 10.0])
print(f'{'Error':>8} | {'L1 Gradient':>14} | {'L2 Gradient':>14}')
print('-' * 42)
for e in errors:
    l1g = np.sign(e)
    l2g = 2 * e
    print(f'{e:>8.2f} | {l1g:>14.3f} | {l2g:>14.3f}')
print('\n✅ L1 gradient is constant — no explosion for large errors')
print('✅ L2 gradient grows with error — fast convergence for small errors')

## Q12. Gradient Penalties in GANs

**Problem**: Vanilla GAN discriminator can have arbitrarily large gradients → training instability.

**Solution**: WGAN-GP adds a penalty: `λ · E[(‖∇D(x̂)‖₂ - 1)²]`

This enforces **Lipschitz continuity** (gradients ≈ 1 everywhere).

In [ ]:
# Q12 CODE: WGAN-GP gradient penalty computation
import numpy as np

def gradient_penalty(real_samples, fake_samples, discriminator_fn, lambda_gp=10):
    """
    WGAN-GP Gradient Penalty:
    Interpolates between real and fake samples,
    then penalizes discriminator gradients that deviate from norm=1
    """
    batch_size = len(real_samples)
    # Random interpolation coefficient
    alpha = np.random.uniform(0, 1, (batch_size, 1))
    # Interpolated samples (between real and fake)
    interpolated = alpha * real_samples + (1 - alpha) * fake_samples

    # Compute discriminator output at interpolated points
    d_interpolated = discriminator_fn(interpolated)

    # Numerical gradient approximation
    eps = 1e-5
    gradients = []
    for i in range(batch_size):
        x = interpolated[i:i+1]
        grad = (discriminator_fn(x + eps) - discriminator_fn(x - eps)) / (2 * eps)
        gradients.append(grad)
    gradients = np.array(gradients)

    # Gradient norm
    gradient_norms = np.sqrt(np.sum(gradients**2, axis=1) + 1e-8)
    # Penalty: (||grad|| - 1)^2
    gp = lambda_gp * np.mean((gradient_norms - 1)**2)
    return gp, gradient_norms

# Toy discriminator: linear function
def toy_discriminator(x):
    return np.sum(x * 2.5, axis=-1, keepdims=True)  # unnormalized scores

np.random.seed(42)
real = np.random.normal(2, 0.5, (8, 3))
fake = np.random.normal(-2, 0.5, (8, 3))

gp, grad_norms = gradient_penalty(real, fake, toy_discriminator, lambda_gp=10)
print('WGAN-GP Gradient Penalty Demo')
print(f'Gradient norms at interpolated points: {grad_norms.round(3)}')
print(f'Mean gradient norm: {grad_norms.mean():.3f} (ideal = 1.0)')
print(f'Gradient Penalty (GP): {gp:.4f}')
print('\n✅ GP penalizes deviations from norm=1, enforcing Lipschitz constraint')

---
# 🔷 SECTION 2: Large Language Models
---

## Q13. Transfer Learning in NLP

**Two phases**:
1. **Pre-training**: Learn language from massive corpus (unsupervised)
2. **Fine-tuning**: Adapt to specific task with small labeled dataset

**Benefits**: Less data needed, faster training, better generalization

In [ ]:
# Q13 CODE: Transfer learning simulation — freeze/unfreeze layers
import numpy as np

class SimpleTransformerLayer:
    def __init__(self, layer_id, frozen=False):
        self.layer_id = layer_id
        self.frozen = frozen
        self.weights = np.random.randn(4, 4) * 0.1  # pre-trained weights
        self.grad_updates = 0

    def update(self, grad, lr=0.001):
        if not self.frozen:
            self.weights -= lr * grad
            self.grad_updates += 1

# Pre-trained model: 6 layers
pretrained_layers = [SimpleTransformerLayer(i) for i in range(6)]

# Fine-tuning strategy: Gradual Unfreezing
# Start by freezing all early layers, only train last layer
def gradual_unfreeze(layers, current_epoch, unfreeze_every=3):
    n_layers = len(layers)
    # Unfreeze from top (last) to bottom (first)
    layers_to_unfreeze = (current_epoch // unfreeze_every) + 1
    for i, layer in enumerate(layers):
        if i >= n_layers - layers_to_unfreeze:
            layer.frozen = False
        else:
            layer.frozen = True
    return layers

# Differential learning rates: lower for earlier layers
def get_layer_lr(base_lr, layer_id, n_layers, decay=0.9):
    """Earlier layers get smaller LR to preserve general knowledge"""
    return base_lr * (decay ** (n_layers - 1 - layer_id))

print('=== Gradual Unfreezing Schedule ===')
for epoch in [0, 3, 6, 9, 12]:
    layers = gradual_unfreeze(pretrained_layers, epoch)
    trainable = [l.layer_id for l in layers if not l.frozen]
    frozen = [l.layer_id for l in layers if l.frozen]
    print(f'  Epoch {epoch:2d}: Trainable={trainable}, Frozen={frozen}')

print('\n=== Differential Learning Rates ===')
base_lr = 0.001
n_layers = 6
for i in range(n_layers):
    lr = get_layer_lr(base_lr, i, n_layers)
    print(f'  Layer {i}: LR = {lr:.6f}  {"← general knowledge, learn slowly" if i==0 else "← task-specific" if i==5 else ""}')

## Q14. GPT vs BERT

| | GPT | BERT |
|---|---|---|
| Architecture | Transformer **Decoder** | Transformer **Encoder** |
| Direction | Left-to-right (causal) | Bidirectional |
| Pre-training | Next token prediction | MLM + NSP |
| Best for | Text generation | Understanding/classification |
| Example task | Write a story | Sentiment analysis |

In [ ]:
# Q14 CODE: Causal (GPT) vs Bidirectional (BERT) attention masks
import numpy as np

def create_causal_mask(seq_len):
    """GPT-style: each token attends only to itself and previous tokens"""
    mask = np.tril(np.ones((seq_len, seq_len)))
    return mask

def create_bidirectional_mask(seq_len):
    """BERT-style: each token attends to ALL tokens"""
    return np.ones((seq_len, seq_len))

def create_mlm_mask(tokens, mask_prob=0.15):
    """BERT Masked Language Modeling: randomly mask 15% of tokens"""
    masked = tokens.copy()
    labels = np.full_like(tokens, -100)  # -100 = ignore in loss
    for i in range(len(tokens)):
        if np.random.rand() < mask_prob:
            labels[i] = tokens[i]   # save true label
            masked[i] = '[MASK]'    # replace with mask token
    return masked, labels

seq_len = 5
print('=== GPT Causal Attention Mask (can only look left) ===')
print('1 = can attend, 0 = cannot attend')
print(create_causal_mask(seq_len).astype(int))

print('\n=== BERT Bidirectional Attention Mask (looks in all directions) ===')
print(create_bidirectional_mask(seq_len).astype(int))

print('\n=== BERT MLM Demo ===')
np.random.seed(7)
sentence = np.array(['The', 'cat', 'sat', 'on', 'mat', 'today'])
masked_sentence, labels = create_mlm_mask(sentence)
print(f'Original:  {" ".join(sentence)}')
print(f'Masked:    {" ".join(masked_sentence)}')
print(f'Predict:   {labels}  (-100 = not masked, string = predict this)')

## Q15. What Problems of RNNs Do Transformers Solve?

| Problem | RNN | Transformer |
|---|---|---|
| Parallelization | Sequential ❌ | Parallel ✅ |
| Long-range dependencies | Vanishing gradients ❌ | Direct attention ✅ |
| Scalability | O(n) but slow | O(n²) but parallelizable ✅ |

In [ ]:
# Q15 CODE: Vanishing gradient in RNNs vs Transformers
import numpy as np

# === RNN: gradient vanishes over long sequences ===
def rnn_gradient_flow(seq_len, W_hh=0.9):
    """Gradient of loss at time t w.r.t. hidden state at time 0"""
    # Simplified: gradient = W_hh^T (matrix power)
    gradients = []
    for t in range(seq_len):
        grad = W_hh ** t   # exponential decay/explosion
        gradients.append(grad)
    return gradients

# === Transformer: attention directly connects any two positions ===
def transformer_attention_distance(seq_len):
    """Every position attends to every other — gradient path length = 1"""
    return {f'pos_0_to_pos_{i}': 1 for i in range(seq_len)}  # always 1 step!

print('=== RNN Gradient Magnitude Over Sequence Length ===')
print('(W_hh=0.9, simulating vanishing gradients)')
rnn_grads = rnn_gradient_flow(seq_len=20, W_hh=0.9)
for t in [0, 4, 9, 14, 19]:
    print(f'  t={t:2d}: gradient magnitude = {rnn_grads[t]:.6f}{" ← vanished!" if rnn_grads[t] < 0.01 else ""}')

print('\n=== Transformer: Constant Gradient Path ===')
attn = transformer_attention_distance(20)
print('  Every pair of positions has path length = 1 (direct attention)')
print('  No vanishing gradient problem!')

print('\n=== Parallelization: RNN vs Transformer ===')
import time
seq = list(range(1000))

# RNN-style: must be sequential
t0 = time.time()
h = 0
for x in seq:
    h = np.tanh(0.5 * h + 0.5 * x * 0.001)
rnn_time = time.time() - t0

# Transformer-style: can be parallel
t0 = time.time()
X = np.array(seq) * 0.001
out = np.tanh(X)  # processed all at once!
tf_time = time.time() - t0

print(f'  RNN sequential processing: {rnn_time*1000:.3f}ms')
print(f'  Transformer parallel processing: {tf_time*1000:.3f}ms')

## Q16. Relative Positional Encoding

**Absolute PE**: Each position gets a fixed encoding. Doesn't generalize to longer sequences.

**Relative PE**: Encodes the *distance* between tokens, not absolute position.

**Used in**: T5, Transformer-XL, RoPE (LLaMA, GPT-NeoX)

In [ ]:
# Q16 CODE: Absolute vs Relative Positional Encodings
import numpy as np

def sinusoidal_positional_encoding(seq_len, d_model):
    """Standard Transformer absolute PE (Vaswani et al. 2017)"""
    PE = np.zeros((seq_len, d_model))
    positions = np.arange(seq_len)[:, np.newaxis]
    dims = np.arange(0, d_model, 2)
    angles = positions / np.power(10000, dims / d_model)
    PE[:, 0::2] = np.sin(angles)
    PE[:, 1::2] = np.cos(angles[:, :d_model//2])
    return PE

def relative_position_bias(seq_len, n_heads=2):
    """T5-style learned relative position bias"""
    # Returns a bias for each (query, key) pair based on their relative distance
    bias = np.zeros((n_heads, seq_len, seq_len))
    for i in range(seq_len):
        for j in range(seq_len):
            dist = i - j  # relative distance
            # Simplified: bias decreases with distance (closer = higher attention)
            for h in range(n_heads):
                bias[h, i, j] = -abs(dist) * 0.1 * (h + 1)
    return bias

seq_len, d_model = 5, 8

abs_pe = sinusoidal_positional_encoding(seq_len, d_model)
rel_bias = relative_position_bias(seq_len, n_heads=2)

print('=== Absolute Positional Encoding (first 4 dims) ===')
for pos in range(seq_len):
    print(f'  Position {pos}: {abs_pe[pos, :4].round(3)}')

print('\n=== Relative Position Bias (Head 0) ===')
print('  Row i, Col j = bias when query=i attends to key=j')
print(rel_bias[0].round(2))
print('\n  Note: diagonal=0 (same position), off-diagonal decreases by distance')

## Q17. Fixed Attention Span Limitations

Vanilla Transformer: O(n²) memory and compute w.r.t sequence length.

**Solutions**: Sparse Attention, Flash Attention, Sliding Window, Longformer

In [ ]:
# Q17 CODE: Attention complexity and sparse attention patterns
import numpy as np

def full_attention_memory(seq_len, d_model=512, bytes_per_param=4):
    """Memory for attention matrix: O(n^2 * d_model)"""
    attn_matrix = seq_len * seq_len * bytes_per_param
    return attn_matrix / (1024**2)  # MB

def sparse_attention_mask(seq_len, window_size=3):
    """Sliding window sparse attention — each token attends to window_size neighbors"""
    mask = np.zeros((seq_len, seq_len))
    for i in range(seq_len):
        start = max(0, i - window_size//2)
        end   = min(seq_len, i + window_size//2 + 1)
        mask[i, start:end] = 1
    return mask

def longformer_attention_mask(seq_len, window_size=2, global_tokens=2):
    """Longformer: sliding window + global tokens (e.g., [CLS])"""
    mask = sparse_attention_mask(seq_len, window_size)
    # Global tokens attend to and are attended by ALL tokens
    for g in range(global_tokens):
        mask[g, :] = 1
        mask[:, g] = 1
    return mask

print('=== Memory Requirements: Full Attention ===')
for n in [512, 1024, 2048, 4096, 8192]:
    mem = full_attention_memory(n)
    print(f'  seq_len={n:5d}: {mem:.2f} MB (just attention matrix)')

print('\n=== Sparse Attention Pattern (seq_len=8, window=3) ===')
sparse = sparse_attention_mask(8, window_size=3)
print(sparse.astype(int))
density = sparse.mean()
print(f'Sparsity: {density:.2%} of entries are active (vs 100% for full attention)')

print('\n=== Longformer Pattern (seq_len=8, window=2, 2 global tokens) ===')
lf = longformer_attention_mask(8, window_size=2, global_tokens=2)
print(lf.astype(int))
print(f'CLS tokens (rows 0-1) attend to everything; others use local window')

## Q18. Self-Attention Mechanism

**Steps**:
1. Compute Q, K, V matrices from input
2. Attention scores = softmax(QKᵀ / √d_k)
3. Output = scores × V

**Intuition**: Each token asks "which other tokens are relevant to me?" (Q) and answers "what do I contain?" (K, V)

In [ ]:
# Q18 CODE: Full Self-Attention from scratch
import numpy as np

def softmax(x, axis=-1):
    x = x - x.max(axis=axis, keepdims=True)  # numerical stability
    exp_x = np.exp(x)
    return exp_x / exp_x.sum(axis=axis, keepdims=True)

def self_attention(X, W_Q, W_K, W_V, mask=None):
    """
    X:    (seq_len, d_model) — input embeddings
    W_Q/K/V: (d_model, d_k) — projection matrices
    Returns: output (seq_len, d_v), attention_weights (seq_len, seq_len)
    """
    d_k = W_K.shape[1]

    Q = X @ W_Q   # (seq_len, d_k)
    K = X @ W_K   # (seq_len, d_k)
    V = X @ W_V   # (seq_len, d_v)

    # Scaled dot-product attention
    scores = Q @ K.T / np.sqrt(d_k)   # (seq_len, seq_len)

    if mask is not None:
        scores = np.where(mask == 0, -1e9, scores)  # mask out future tokens

    attn_weights = softmax(scores)     # (seq_len, seq_len)
    output = attn_weights @ V          # (seq_len, d_v)
    return output, attn_weights

# Example: 4 tokens, d_model=8, d_k=4
np.random.seed(42)
seq_len, d_model, d_k = 4, 8, 4

X   = np.random.randn(seq_len, d_model)
W_Q = np.random.randn(d_model, d_k) * 0.1
W_K = np.random.randn(d_model, d_k) * 0.1
W_V = np.random.randn(d_model, d_k) * 0.1

# Bidirectional (BERT-style)
output_bidir, attn_bidir = self_attention(X, W_Q, W_K, W_V)

# Causal (GPT-style)
causal_mask = np.tril(np.ones((seq_len, seq_len)))
output_causal, attn_causal = self_attention(X, W_Q, W_K, W_V, mask=causal_mask)

print('=== Bidirectional Attention Weights (BERT-style) ===')
print('Each row: how much token i attends to each other token')
print(attn_bidir.round(3))
print('\n=== Causal Attention Weights (GPT-style) ===')
print('Upper triangle = 0 (cannot attend to future tokens)')
print(attn_causal.round(3))
print('\n✅ Row sums = 1.0 (softmax):', attn_bidir.sum(axis=1).round(3))

## Q19. Multi-Head Attention

**Why**: Single attention head can only model one type of relationship at a time.

**Multi-head**: Run H attention heads in parallel → each learns different relationships (syntactic, semantic, etc.) → concatenate outputs.

In [ ]:
# Q19 CODE: Multi-Head Attention from scratch
import numpy as np

def softmax(x):
    x = x - x.max(axis=-1, keepdims=True)
    return np.exp(x) / np.exp(x).sum(axis=-1, keepdims=True)

def multi_head_attention(X, n_heads=2, d_model=8):
    """
    X: (seq_len, d_model)
    Returns: output (seq_len, d_model)
    """
    d_k = d_model // n_heads
    seq_len = X.shape[0]

    head_outputs = []
    attn_weights_per_head = []

    for h in range(n_heads):
        # Each head has its own learned projection matrices
        W_Q = np.random.randn(d_model, d_k) * 0.1
        W_K = np.random.randn(d_model, d_k) * 0.1
        W_V = np.random.randn(d_model, d_k) * 0.1

        Q = X @ W_Q  # (seq_len, d_k)
        K = X @ W_K
        V = X @ W_V

        scores = softmax(Q @ K.T / np.sqrt(d_k))
        head_out = scores @ V  # (seq_len, d_k)

        head_outputs.append(head_out)
        attn_weights_per_head.append(scores)

    # Concatenate all heads: (seq_len, d_model)
    concat = np.concatenate(head_outputs, axis=-1)

    # Final output projection
    W_O = np.random.randn(d_model, d_model) * 0.1
    output = concat @ W_O

    return output, attn_weights_per_head

np.random.seed(42)
seq_len, d_model, n_heads = 5, 8, 2
X = np.random.randn(seq_len, d_model)

output, attn_per_head = multi_head_attention(X, n_heads=n_heads, d_model=d_model)

print(f'Input shape: {X.shape}')
print(f'Output shape: {output.shape}  (same as input — information enriched!)')
print(f'\nAttention weights — Head 0 (may capture syntactic relationships):')
print(attn_per_head[0].round(3))
print(f'\nAttention weights — Head 1 (may capture semantic relationships):')
print(attn_per_head[1].round(3))
print('\n✅ Different heads focus on different patterns in the data')

## Q20. RLHF (Reinforcement Learning from Human Feedback)

**Steps**:
1. Pre-train LLM on large corpus
2. Collect human preference data (which response is better?)
3. Train a **Reward Model** on these preferences
4. Fine-tune LLM with PPO using reward model as reward signal

In [ ]:
# Q20 CODE: RLHF reward model and PPO objective simulation
import numpy as np

# === Step 1: Reward Model Training ===
# Bradley-Terry model: P(A > B) = sigma(r(A) - r(B))
def preference_loss(reward_A, reward_B):
    """Loss for human preferring A over B: -log(sigma(r_A - r_B))"""
    def sigmoid(x): return 1 / (1 + np.exp(-x))
    return -np.log(sigmoid(reward_A - reward_B) + 1e-8)

# === Step 2: PPO Clip Objective ===
def ppo_clip_loss(old_log_probs, new_log_probs, advantages, eps=0.2):
    """
    PPO-Clip: prevents too-large policy updates
    r(theta) = pi_new / pi_old = exp(log_pi_new - log_pi_old)
    Loss = -min(r*A, clip(r, 1-eps, 1+eps)*A)
    """
    ratios = np.exp(new_log_probs - old_log_probs)
    clipped_ratios = np.clip(ratios, 1 - eps, 1 + eps)
    policy_loss = -np.minimum(ratios * advantages, clipped_ratios * advantages)
    return policy_loss.mean(), ratios

# === Step 3: KL Penalty (to prevent drifting too far from base model) ===
def kl_penalty(new_log_probs, ref_log_probs, beta=0.1):
    """Penalize KL divergence from reference (pre-trained) model"""
    kl = new_log_probs - ref_log_probs
    return beta * kl.mean()

np.random.seed(42)

# Reward model comparison
print('=== Reward Model: Human Preference Learning ===')
pairs = [
    ('Helpful, safe response', 3.5, 'Harmful response', -1.0),
    ('Detailed answer', 2.0, 'Vague answer', 0.5),
]
for a_text, r_a, b_text, r_b in pairs:
    loss = preference_loss(r_a, r_b)
    print(f'  Preferred: "{a_text}" (r={r_a}) over "{b_text}" (r={r_b})')
    print(f'  Preference loss: {loss:.4f}')

print('\n=== PPO-Clip Objective ===')
old_log_probs = np.array([-0.5, -1.0, -0.3, -0.8])
new_log_probs = np.array([-0.4, -0.8, -0.5, -0.6])
advantages    = np.array([1.2, 0.5, -0.3, 0.8])  # positive = good action

loss, ratios = ppo_clip_loss(old_log_probs, new_log_probs, advantages)
kl = kl_penalty(new_log_probs, old_log_probs)

print(f'Policy ratios (new/old): {ratios.round(3)}')
print(f'PPO Clip Loss: {loss:.4f}')
print(f'KL Penalty: {kl:.4f}')
print(f'Total RLHF Loss = PPO Loss + KL Penalty = {loss + kl:.4f}')

## Q21. Positional Encoding

Without PE, `"The cat ate the fish"` and `"The fish ate the cat"` look identical to the model.

**Sinusoidal PE**: `PE(pos, 2i) = sin(pos/10000^(2i/d))`, `PE(pos, 2i+1) = cos(...)`

In [ ]:
# Q21 CODE: Positional Encoding & why position matters
import numpy as np

def get_pe(seq_len, d_model):
    pe = np.zeros((seq_len, d_model))
    for pos in range(seq_len):
        for i in range(0, d_model, 2):
            pe[pos, i]   = np.sin(pos / (10000 ** (i / d_model)))
            if i+1 < d_model:
                pe[pos, i+1] = np.cos(pos / (10000 ** (i / d_model)))
    return pe

# Why position matters: same words, different meaning
words = ['The', 'cat', 'ate', 'the', 'fish']
d_model = 8

# Fake word embeddings (same for same word)
word_emb = {
    'The': np.array([1, 0, 1, 0, 1, 0, 1, 0], dtype=float),
    'the': np.array([1, 0, 1, 0, 1, 0, 1, 0], dtype=float),
    'cat': np.array([0, 1, 0, 1, 0, 0, 0, 1], dtype=float),
    'ate': np.array([0, 0, 1, 0, 0, 1, 0, 0], dtype=float),
    'fish': np.array([0, 1, 0, 0, 1, 0, 1, 0], dtype=float),
}

pe = get_pe(len(words), d_model)

print('=== Without Positional Encoding ===')
print('"The" at position 0 and "the" at position 3 are IDENTICAL:')
print(f'  Pos 0 (The):  {word_emb["The"]}')
print(f'  Pos 3 (the):  {word_emb["the"]}')

print('\n=== With Positional Encoding Added ===')
print('Now they are different!')
emb_pos0 = word_emb['The'] + pe[0]
emb_pos3 = word_emb['the'] + pe[3]
print(f'  Pos 0 (The+PE):  {emb_pos0.round(3)}')
print(f'  Pos 3 (the+PE):  {emb_pos3.round(3)}')

# Cosine similarity: lower = more different
cos_sim = np.dot(emb_pos0, emb_pos3) / (np.linalg.norm(emb_pos0) * np.linalg.norm(emb_pos3))
print(f'  Cosine similarity (with PE): {cos_sim:.4f}  (1.0 = identical, lower = more different)')

## Q22. Catastrophic Forgetting

When fine-tuning on Task B, a model *forgets* Task A.

**Solutions**: Elastic Weight Consolidation (EWC), Replay, LoRA (Low-Rank Adaptation)

In [ ]:
# Q22 CODE: Elastic Weight Consolidation (EWC) to prevent catastrophic forgetting
import numpy as np

# EWC adds a regularization penalty:
# L_EWC = L_new + (lambda/2) * Σ_i F_i * (theta_i - theta_i*)^2
# Where F_i = Fisher information (how important weight i was for old task)

def compute_fisher_diagonal(gradients_list):
    """Approximate diagonal of Fisher information matrix as E[grad^2]"""
    squared_grads = [g**2 for g in gradients_list]
    return np.mean(squared_grads, axis=0)

def ewc_penalty(current_params, old_params, fisher, lambda_ewc=1000):
    """Penalize deviation from old parameters, weighted by importance (Fisher)"""
    diff = current_params - old_params
    return (lambda_ewc / 2) * np.sum(fisher * diff**2)

def total_ewc_loss(task_loss, current_params, old_params, fisher, lambda_ewc=1000):
    return task_loss + ewc_penalty(current_params, old_params, fisher, lambda_ewc)

# Simulate training on Task A, then fine-tuning on Task B with EWC
np.random.seed(42)
n_params = 10

# After training on Task A
theta_star = np.random.randn(n_params)   # optimal params for Task A

# Fisher diagonal: high = important for Task A, low = not important
fisher = np.abs(np.random.randn(n_params))
fisher[0:3] *= 5  # first 3 params are very important for Task A

print('Fisher Information (importance for Task A):')
print(np.round(fisher, 2))
print('↑ High values = weight is critical for Task A')

# Two update scenarios:
# 1. Without EWC — drifts freely
drift_without_ewc = theta_star + np.random.randn(n_params) * 2.0

# 2. With EWC — important weights stay close to theta_star
task_b_loss = 0.5  # loss on new task

ewc_pen = ewc_penalty(drift_without_ewc, theta_star, fisher)
print(f'\nEWC Penalty if we drift freely: {ewc_pen:.2f}')
print('→ EWC gradient will push important weights back toward theta_star!')
print('\n✅ EWC protects Task A performance while learning Task B')

---
# 🔷 SECTION 3: Multimodal Models
---

## Q23. Cross-Modal Attention in Vision-Language Models

**Cross-attention**: Text query attends to image regions (or vice versa).
- Used in VisualBERT, CLIP, Flamingo
- Enables tasks like VQA, image captioning, image-text retrieval

In [ ]:
# Q23 CODE: Cross-modal attention (text queries attending to image regions)
import numpy as np

def softmax(x):
    x -= x.max(axis=-1, keepdims=True)
    return np.exp(x) / np.exp(x).sum(axis=-1, keepdims=True)

def cross_modal_attention(text_queries, image_features):
    """
    text_queries: (n_words, d_model) — text tokens as queries
    image_features: (n_regions, d_model) — image regions as keys/values
    Returns: enriched text features that incorporate relevant image info
    """
    d_k = text_queries.shape[1]
    # Attention scores: each text token vs each image region
    scores = text_queries @ image_features.T / np.sqrt(d_k)  # (n_words, n_regions)
    attn_weights = softmax(scores)
    # Weighted sum of image features
    enriched_text = attn_weights @ image_features  # (n_words, d_model)
    return enriched_text, attn_weights

np.random.seed(42)
d_model = 6

# Simulate: question "What color is the cat?" (3 word tokens)
text_tokens = ['What', 'color', 'cat?']
text_query = np.array([
    [0.1, 0.2, 0.3, 0.1, 0.0, 0.1],  # 'What'
    [0.9, 0.1, 0.0, 0.5, 0.2, 0.1],  # 'color'  ← should attend to colored regions
    [0.2, 0.8, 0.1, 0.3, 0.7, 0.2],  # 'cat'    ← should attend to cat region
])

# Image regions: background, cat, sky, tree
image_regions = ['background', 'cat', 'sky', 'tree']
image_feats = np.array([
    [0.0, 0.1, 0.0, 0.1, 0.0, 0.1],  # background (low signal)
    [0.1, 0.9, 0.2, 0.3, 0.8, 0.3],  # cat         (orange/furry)
    [0.3, 0.1, 0.8, 0.1, 0.2, 0.7],  # sky         (blue)
    [0.2, 0.5, 0.1, 0.8, 0.1, 0.2],  # tree        (green)
])

enriched, attn = cross_modal_attention(text_query, image_feats)

print('=== Cross-Modal Attention: Text tokens attending to Image regions ===')
print(f'{'Token':>12} | ' + ' | '.join(f'{r:>12}' for r in image_regions))
print('-' * 68)
for i, token in enumerate(text_tokens):
    weights = ' | '.join(f'{attn[i,j]:>12.3f}' for j in range(len(image_regions)))
    print(f'{token:>12} | {weights}')

print('\n✅ "cat?" token attends most to the cat image region!')

## Q24. Loss Functions for Image Synthesis

1. **Pixel-wise (MSE/MAE)**: Direct per-pixel comparison — simple but blurry
2. **Adversarial (GAN)**: Discriminator-based realism — sharp but unstable
3. **Perceptual**: Deep feature comparison — captures style & content

In [ ]:
# Q24 CODE: Image synthesis loss functions
import numpy as np

def pixel_mse_loss(generated, target):
    """Per-pixel MSE — tends to produce blurry images"""
    return np.mean((generated - target)**2)

def pixel_mae_loss(generated, target):
    """Per-pixel MAE — sharper edges than MSE"""
    return np.mean(np.abs(generated - target))

def perceptual_loss(gen_features, target_features):
    """
    Compares feature maps from a pre-trained network (e.g. VGG layer 3)
    Captures semantic similarity, not pixel-perfect matching
    """
    return np.mean((gen_features - target_features)**2)

def adversarial_generator_loss(discriminator_on_fake):
    """Generator wants discriminator to output 1 (real) for fake images"""
    return -np.mean(np.log(discriminator_on_fake + 1e-8))

def combined_loss(gen, target, gen_feats, target_feats, d_fake,
                  lambda_pixel=10, lambda_perceptual=1, lambda_adv=1):
    pix   = pixel_mae_loss(gen, target)
    perc  = perceptual_loss(gen_feats, target_feats)
    adv   = adversarial_generator_loss(d_fake)
    total = lambda_pixel * pix + lambda_perceptual * perc + lambda_adv * adv
    return total, pix, perc, adv

np.random.seed(42)
H, W, C = 4, 4, 3  # small 4x4 image for demo
target_img  = np.random.rand(H, W, C)
perfect_gen = target_img + np.random.randn(H,W,C) * 0.05  # nearly perfect
blurry_gen  = np.ones_like(target_img) * target_img.mean() # blurry (mean color)

# Simulated deep features (normally from VGG)
target_feats  = np.random.randn(64)
perfect_feats = target_feats + np.random.randn(64) * 0.1
blurry_feats  = target_feats + np.random.randn(64) * 2.0  # blurry = different features

d_perfect = np.array([0.9])  # discriminator thinks it's real
d_blurry  = np.array([0.1])  # discriminator knows it's fake

print('=== Image Synthesis Loss Comparison ===')
print(f'\nPixel MSE  — Perfect: {pixel_mse_loss(perfect_gen, target_img):.4f} | Blurry: {pixel_mse_loss(blurry_gen, target_img):.4f}')
print(f'Pixel MAE  — Perfect: {pixel_mae_loss(perfect_gen, target_img):.4f} | Blurry: {pixel_mae_loss(blurry_gen, target_img):.4f}')
print(f'Perceptual — Perfect: {perceptual_loss(perfect_feats, target_feats):.4f} | Blurry: {perceptual_loss(blurry_feats, target_feats):.4f}')
print(f'Adversarial— Perfect: {adversarial_generator_loss(d_perfect):.4f} | Blurry: {adversarial_generator_loss(d_blurry):.4f}')

## Q25. CLIP: Contrastive Language-Image Pre-training

**CLIP trains by**: Maximizing similarity between matching (image, text) pairs; minimizing it for non-matching pairs.

**Loss**: InfoNCE / NT-Xent contrastive loss

In [ ]:
# Q25 CODE: CLIP-style contrastive loss
import numpy as np

def l2_normalize(x, axis=-1):
    return x / (np.linalg.norm(x, axis=axis, keepdims=True) + 1e-8)

def clip_contrastive_loss(image_embeddings, text_embeddings, temperature=0.07):
    """
    CLIP InfoNCE loss:
    - image_embeddings: (N, D)
    - text_embeddings:  (N, D)
    - Diagonal = matching pairs, off-diagonal = non-matching
    """
    # Normalize embeddings to unit sphere
    img_emb = l2_normalize(image_embeddings)
    txt_emb = l2_normalize(text_embeddings)

    # Similarity matrix: (N, N)
    logits = (img_emb @ txt_emb.T) / temperature

    # Labels: diagonal (i,i) should be highest
    N = len(image_embeddings)
    labels = np.arange(N)

    def cross_entropy_loss(logits_matrix, labels):
        # Softmax cross-entropy
        log_probs = logits_matrix - np.log(np.exp(logits_matrix).sum(axis=1, keepdims=True) + 1e-8)
        return -np.mean(log_probs[np.arange(N), labels])

    loss_img2txt = cross_entropy_loss(logits, labels)   # image → text
    loss_txt2img = cross_entropy_loss(logits.T, labels) # text → image
    return (loss_img2txt + loss_txt2img) / 2, logits

np.random.seed(42)
N, D = 4, 8  # 4 image-text pairs, 8-dim embeddings

# Well-aligned embeddings (matching pairs are similar)
base = np.random.randn(N, D)
good_img_emb = base + np.random.randn(N, D) * 0.1
good_txt_emb = base + np.random.randn(N, D) * 0.1

# Misaligned embeddings (random)
bad_img_emb = np.random.randn(N, D)
bad_txt_emb = np.random.randn(N, D)

good_loss, good_sim = clip_contrastive_loss(good_img_emb, good_txt_emb)
bad_loss,  bad_sim  = clip_contrastive_loss(bad_img_emb, bad_txt_emb)

print('=== CLIP Contrastive Loss ===')
print(f'Well-aligned pairs loss: {good_loss:.4f}  ← low (model knows pairs)')
print(f'Misaligned pairs loss:   {bad_loss:.4f}  ← high (model confused)')
print('\nSimilarity matrix (well-aligned) — diagonal should be highest:')
print(good_sim.round(2))
print('\n✅ CLIP learns to push matching pairs together, non-matching apart')

---
# 🔷 SECTION 4: Embeddings
---

## Q26. What are Embeddings?

Dense, low-dimensional vectors that represent high-dimensional data.

**Key property**: Similar items → similar vectors (close in embedding space)

In [ ]:
# Q26 CODE: Word embeddings — cosine similarity & analogies
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b) + 1e-8)

def find_nearest(query_vec, embeddings, top_k=3):
    sims = {word: cosine_similarity(query_vec, vec)
            for word, vec in embeddings.items()}
    return sorted(sims.items(), key=lambda x: -x[1])[:top_k]

# Toy word embeddings (8-dim, manually crafted to show semantic relationships)
# Dims: [royal, male, female, animal, country, food, size, age]
word_embeddings = {
    'king':    np.array([1, 1, 0, 0, 0, 0, 1, 0.5], dtype=float),
    'queen':   np.array([1, 0, 1, 0, 0, 0, 1, 0.5], dtype=float),
    'man':     np.array([0, 1, 0, 0, 0, 0, 0.5, 0.5], dtype=float),
    'woman':   np.array([0, 0, 1, 0, 0, 0, 0.5, 0.5], dtype=float),
    'prince':  np.array([1, 1, 0, 0, 0, 0, 0.5, 0.2], dtype=float),
    'dog':     np.array([0, 0, 0, 1, 0, 0, 0.5, 0.3], dtype=float),
    'cat':     np.array([0, 0, 0, 1, 0, 0, 0.3, 0.3], dtype=float),
    'france':  np.array([0, 0, 0, 0, 1, 0, 0.8, 0.9], dtype=float),
    'paris':   np.array([0, 0, 0, 0, 0.8, 0, 0.5, 0.9], dtype=float),
}

print('=== Word Similarity ===')
pairs = [('king', 'queen'), ('king', 'man'), ('dog', 'cat'), ('king', 'dog')]
for w1, w2 in pairs:
    sim = cosine_similarity(word_embeddings[w1], word_embeddings[w2])
    print(f'  sim({w1}, {w2}) = {sim:.3f}')

print('\n=== Word Analogy: king - man + woman ≈ ? ===')
analogy_vec = word_embeddings['king'] - word_embeddings['man'] + word_embeddings['woman']
nearest = find_nearest(analogy_vec, word_embeddings, top_k=3)
print('Top results:')
for word, sim in nearest:
    print(f'  {word}: {sim:.3f}')
print('✅ Expected: "queen" should be closest!')

## Q27. Triplet Loss

**Goal**: Learn embeddings where similar items are close, dissimilar items are far.

**Formula**: `L = max(0, d(anchor, positive) - d(anchor, negative) + margin)`

**Margin**: Minimum desired gap between positive and negative distances.

In [ ]:
# Q27 CODE: Triplet Loss + Contrastive Loss
import numpy as np

def euclidean_distance(a, b):
    return np.sqrt(np.sum((a - b)**2) + 1e-8)

def triplet_loss(anchor, positive, negative, margin=1.0):
    """
    Loss = max(0, d(A,P) - d(A,N) + margin)
    Forces: d(A,N) > d(A,P) + margin
    """
    d_pos = euclidean_distance(anchor, positive)
    d_neg = euclidean_distance(anchor, negative)
    loss  = max(0, d_pos - d_neg + margin)
    return loss, d_pos, d_neg

def contrastive_loss(emb1, emb2, label, margin=1.0):
    """
    label=1 → similar pair → minimize distance
    label=0 → dissimilar → maximize distance (up to margin)
    """
    dist = euclidean_distance(emb1, emb2)
    loss = label * dist**2 + (1 - label) * max(0, margin - dist)**2
    return loss, dist

np.random.seed(42)

# Before training: random embeddings
anchor   = np.array([0.5, 0.3, -0.2, 0.8])
positive = np.array([0.4, 0.4, -0.1, 0.9])  # same class, should be close
negative = np.array([-0.5, 1.0, 0.8, -0.2]) # different class, should be far

print('=== Triplet Loss ===')
for margin in [0.5, 1.0, 2.0]:
    loss, d_pos, d_neg = triplet_loss(anchor, positive, negative, margin)
    print(f'  margin={margin:.1f}: d(A,P)={d_pos:.3f}, d(A,N)={d_neg:.3f}, loss={loss:.4f}'
          + (' ← loss=0, constraint satisfied!' if loss==0 else ' ← need to push N further'))

print('\n=== Contrastive Loss ===')
similar_pairs   = [(anchor, positive, 1), (anchor, negative, 0)]
for e1, e2, lbl in similar_pairs:
    loss, dist = contrastive_loss(e1, e2, lbl, margin=1.0)
    pair_type = 'similar' if lbl == 1 else 'dissimilar'
    print(f'  {pair_type} pair: dist={dist:.3f}, loss={loss:.4f}')

print('\n=== Margin Effect on Embedding Space ===')
print('Larger margin → stricter separation → more discriminative embeddings')
for m in [0.1, 0.5, 1.0, 2.0]:
    loss, _, _ = triplet_loss(anchor, positive, negative, margin=m)
    print(f'  margin={m}: loss={loss:.4f}')

## Q28. Quantization of Embeddings

**Reduces memory** by lowering numerical precision:
- FP32 (4 bytes) → FP16 (2 bytes) → INT8 (1 byte)
- Used to deploy large models on mobile/edge devices

In [ ]:
# Q28 CODE: Quantization — memory saving & quality preservation
import numpy as np

def quantize_int8(embeddings):
    """Quantize FP32 embeddings to INT8 range [-128, 127]"""
    min_val = embeddings.min()
    max_val = embeddings.max()
    scale   = (max_val - min_val) / 255
    zero_point = -128 - min_val / scale
    # Quantize
    q = np.round(embeddings / scale + zero_point).astype(np.int8)
    return q, scale, zero_point

def dequantize_int8(q, scale, zero_point):
    """Recover approximate FP32 values from INT8"""
    return (q.astype(np.float32) - zero_point) * scale

def memory_usage(arr):
    return arr.nbytes

np.random.seed(42)
n_embeddings = 10000
d_model = 768  # BERT-base size

# FP32 embeddings
fp32_embeddings = np.random.randn(n_embeddings, d_model).astype(np.float32)

# FP16
fp16_embeddings = fp32_embeddings.astype(np.float16)

# INT8
int8_embeddings, scale, zp = quantize_int8(fp32_embeddings)
dequantized = dequantize_int8(int8_embeddings, scale, zp)

# Quality: cosine similarity between original and quantized
sample_orig = fp32_embeddings[0]
sample_dq   = dequantized[0]
cos_sim = np.dot(sample_orig, sample_dq) / (np.linalg.norm(sample_orig) * np.linalg.norm(sample_dq))

print('=== Quantization: Memory vs Quality ===')
print(f'Vocab: {n_embeddings:,} embeddings × {d_model} dims')
print(f'FP32 memory:  {memory_usage(fp32_embeddings)/1024**2:.1f} MB')
print(f'FP16 memory:  {memory_usage(fp16_embeddings)/1024**2:.1f} MB  ({memory_usage(fp16_embeddings)/memory_usage(fp32_embeddings):.0%} of FP32)')
print(f'INT8 memory:  {memory_usage(int8_embeddings)/1024**2:.1f} MB  ({memory_usage(int8_embeddings)/memory_usage(fp32_embeddings):.0%} of FP32)')
print(f'\nINT8 Quality (cosine similarity with FP32): {cos_sim:.6f}')
print(f'Quantization error (MSE): {np.mean((fp32_embeddings[0] - dequantized[0])**2):.6f}')
print('\n✅ INT8 uses 4x less memory with minimal quality loss!')

## Q29. Nearest Neighbor Search for Embeddings (ANN)

**Problem**: Brute-force search in database of millions of embeddings is too slow.

**Solution**: Approximate Nearest Neighbors (ANN):
- LSH (Locality Sensitive Hashing)
- HNSW (Hierarchical Navigable Small World)
- FAISS (Facebook AI Similarity Search)

In [ ]:
# Q29 CODE: Brute-force vs LSH approximate nearest neighbors
import numpy as np
import time

# === Brute Force KNN ===
def brute_force_knn(query, database, k=5):
    """O(n*d) — exact but slow for large n"""
    distances = np.linalg.norm(database - query, axis=1)
    indices = np.argsort(distances)[:k]
    return indices, distances[indices]

# === Locality Sensitive Hashing (simplified) ===
class SimpleLSH:
    def __init__(self, d, n_hash_functions=8):
        self.n_hf = n_hash_functions
        # Random hyperplanes for binary hashing
        self.hyperplanes = np.random.randn(n_hash_functions, d)
        self.buckets = {}

    def hash(self, x):
        """Projects x onto hyperplanes, returns binary hash code"""
        projections = self.hyperplanes @ x
        bits = (projections > 0).astype(int)
        return tuple(bits)

    def index(self, database):
        """Index all database vectors"""
        for i, vec in enumerate(database):
            h = self.hash(vec)
            self.buckets.setdefault(h, []).append(i)

    def query(self, q, k=5, database=None):
        """Find approximate nearest neighbors via bucket lookup"""
        h = self.hash(q)
        candidates = self.buckets.get(h, [])
        if not candidates or database is None:
            return [], []
        cand_vecs = database[candidates]
        dists = np.linalg.norm(cand_vecs - q, axis=1)
        sorted_idx = np.argsort(dists)[:k]
        return [candidates[i] for i in sorted_idx], dists[sorted_idx]

np.random.seed(42)
N, D = 10000, 64
database = np.random.randn(N, D).astype(np.float32)
query = np.random.randn(D).astype(np.float32)

# Brute force
t0 = time.time()
bf_idx, bf_dist = brute_force_knn(query, database, k=5)
bf_time = time.time() - t0

# LSH
lsh = SimpleLSH(D, n_hash_functions=10)
lsh.index(database)
t0 = time.time()
lsh_idx, lsh_dist = lsh.query(query, k=5, database=database)
lsh_time = time.time() - t0

print(f'=== Nearest Neighbor Search on {N} vectors, {D}-dim ===')
print(f'Brute Force: {bf_time*1000:.2f}ms | Top-5 indices: {list(bf_idx)}')
print(f'LSH Approx:  {lsh_time*1000:.2f}ms | Top-5 indices: {lsh_idx}')
print(f'Speedup:     {bf_time/max(lsh_time, 1e-6):.1f}x')
print(f'\nLSH bucket size: {len(lsh.buckets.get(lsh.hash(query), []))} candidates (vs {N} brute force)')
print('\n✅ LSH reduces search space dramatically — trade exact for speed')

---
# 🔷 SECTION 5: Training, Inference & Evaluation
---

## Q30. Evaluation Metrics for LLM Generation

1. **Perplexity**: How surprised is the model? Lower = better
2. **BLEU**: N-gram overlap with reference (translation)
3. **ROUGE**: Recall-based n-gram overlap (summarization)
4. **BERTScore**: Semantic similarity via contextual embeddings
5. **Human Evaluation**: Gold standard

In [ ]:
# Q30 CODE: Perplexity, BLEU, ROUGE from scratch
import numpy as np
from collections import Counter
import math

# === PERPLEXITY ===
def perplexity(log_probs):
    """PP = exp(-1/N * Σ log P(w_i))"""
    return math.exp(-np.mean(log_probs))

# === BLEU Score (simplified) ===
def bleu_score(hypothesis, reference, max_n=4):
    """Simplified BLEU: geometric mean of n-gram precisions × brevity penalty"""
    hyp = hypothesis.split()
    ref = reference.split()

    precisions = []
    for n in range(1, max_n + 1):
        if len(hyp) < n:
            break
        hyp_ngrams = Counter([tuple(hyp[i:i+n]) for i in range(len(hyp)-n+1)])
        ref_ngrams = Counter([tuple(ref[i:i+n]) for i in range(len(ref)-n+1)])
        clip_count = sum(min(count, ref_ngrams.get(gram, 0))
                        for gram, count in hyp_ngrams.items())
        precision = clip_count / max(len(hyp) - n + 1, 1)
        precisions.append(precision)

    if not precisions or any(p == 0 for p in precisions):
        return 0.0

    # Brevity penalty
    bp = min(1.0, math.exp(1 - len(ref) / max(len(hyp), 1)))
    log_avg = sum(math.log(p) for p in precisions) / len(precisions)
    return bp * math.exp(log_avg)

# === ROUGE-N Score ===
def rouge_n(hypothesis, reference, n=1):
    hyp = hypothesis.split()
    ref = reference.split()
    def get_ngrams(tokens, n):
        return Counter([tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)])
    hyp_ngrams = get_ngrams(hyp, n)
    ref_ngrams = get_ngrams(ref, n)
    overlap = sum(min(count, hyp_ngrams.get(gram, 0))
                  for gram, count in ref_ngrams.items())
    recall    = overlap / max(sum(ref_ngrams.values()), 1)
    precision = overlap / max(sum(hyp_ngrams.values()), 1)
    f1 = 2 * recall * precision / max(recall + precision, 1e-8)
    return {'precision': precision, 'recall': recall, 'f1': f1}

# Demo
reference  = "The cat sat on the mat today"
hyp_good   = "The cat sat on the mat today"  # perfect
hyp_decent = "A cat was sitting on the mat"
hyp_bad    = "Dog runs fast in park jumping"

print('=== BLEU Scores ===')
for name, hyp in [('Perfect', hyp_good), ('Decent', hyp_decent), ('Bad', hyp_bad)]:
    score = bleu_score(hyp, reference)
    print(f'  {name:8s}: {score:.4f}')

print('\n=== ROUGE-1 Scores ===')
for name, hyp in [('Perfect', hyp_good), ('Decent', hyp_decent), ('Bad', hyp_bad)]:
    scores = rouge_n(hyp, reference, n=1)
    print(f'  {name:8s}: P={scores["precision"]:.2f}, R={scores["recall"]:.2f}, F1={scores["f1"]:.2f}')

print('\n=== Perplexity ===')
log_probs_good = np.array([-0.2, -0.3, -0.1, -0.4, -0.2])  # good model
log_probs_bad  = np.array([-2.0, -3.5, -2.8, -4.0, -3.0])  # bad model
print(f'  Good model perplexity: {perplexity(log_probs_good):.2f}')
print(f'  Bad model perplexity:  {perplexity(log_probs_bad):.2f}')
print('  ↑ Lower perplexity = model is less surprised = better language model')

## Q31. Hallucination in LLMs

**Hallucination**: Model confidently states false or fabricated information.

**Detection methods**:
1. Log probability analysis (low prob = uncertain)
2. SelfCheckGPT (consistency across multiple samples)
3. Factual verification against knowledge base

In [ ]:
# Q31 CODE: Hallucination detection via uncertainty and consistency
import numpy as np

# === Method 1: Sequence Log Probability as Confidence Proxy ===
def sequence_log_prob(token_log_probs):
    """Length-normalized log probability — lower = less confident"""
    return np.mean(token_log_probs)

# === Method 2: SelfCheckGPT — Check consistency across N samples ===
def selfcheck_consistency(samples):
    """
    If model is certain → multiple samples agree → high consistency
    If hallucinating → samples vary wildly → low consistency
    Here we use token overlap as proxy for semantic consistency
    """
    if len(samples) < 2:
        return 1.0

    pairwise_sims = []
    for i in range(len(samples)):
        for j in range(i+1, len(samples)):
            s1 = set(samples[i].lower().split())
            s2 = set(samples[j].lower().split())
            if not s1 or not s2:
                continue
            jaccard = len(s1 & s2) / len(s1 | s2)
            pairwise_sims.append(jaccard)

    return np.mean(pairwise_sims) if pairwise_sims else 0.0

# === Method 3: Factual Consistency Check ===
def fact_check_score(generated_facts, verified_facts):
    """What fraction of generated facts are in the verified knowledge base?"""
    if not generated_facts:
        return 0.0
    correct = sum(1 for f in generated_facts if f in verified_facts)
    return correct / len(generated_facts)

# Demo scenarios
print('=== Hallucination Detection Demo ===')

# Factual response
factual_log_probs = np.array([-0.1, -0.2, -0.15, -0.3, -0.1])
# Hallucinated response (model is uncertain)
halluc_log_probs = np.array([-2.5, -3.1, -2.8, -3.5, -2.9])

print('\n1. Log Probability (lower = less confident = likely hallucination):')
print(f'   Factual response:      {sequence_log_prob(factual_log_probs):.3f}')
print(f'   Hallucinated response: {sequence_log_prob(halluc_log_probs):.3f}  ← very low!')

print('\n2. SelfCheckGPT Consistency (higher = more consistent = less hallucination):')
consistent_samples = [
    "Paris is the capital of France and has the Eiffel Tower",
    "Paris is France capital city with the Eiffel Tower landmark",
    "The capital of France is Paris known for the Eiffel Tower"
]
inconsistent_samples = [
    "The Eiffel Tower was built in 1650 by Napoleon",
    "Eiffel Tower construction started in 1920 as a monument",
    "The tower was completed in 1799 by architect Gustave"
]

c_score = selfcheck_consistency(consistent_samples)
i_score = selfcheck_consistency(inconsistent_samples)
print(f'   Factual claim consistency:    {c_score:.3f}  ← high = trustworthy')
print(f'   Hallucinated claim consistency:{i_score:.3f}  ← low = likely hallucination!')

print('\n3. Fact Check Score:')
knowledge_base = {'Paris is capital of France', 'Eiffel Tower built 1889',
                  'Gustave Eiffel designed it', 'Tower is 330 meters tall'}
generated_good = ['Paris is capital of France', 'Eiffel Tower built 1889', 'Tower is 330 meters tall']
generated_bad  = ['Eiffel Tower built 1650', 'Napoleon built the tower', 'Tower is 600 meters tall']
print(f'   Accurate generation:     {fact_check_score(generated_good, knowledge_base):.2%}')
print(f'   Hallucinated generation: {fact_check_score(generated_bad, knowledge_base):.2%}')

## Q32. Mixture of Experts (MoE)

**Architecture**: Multiple specialized sub-networks (experts) + a gating network that selects which experts handle each input.

**Benefits**: Scale model capacity without increasing inference compute (sparse activation).

**Examples**: Mixtral 8x7B, GPT-4 (rumored), Switch Transformer

In [ ]:
# Q32 CODE: Mixture of Experts implementation
import numpy as np

def softmax(x):
    x = x - x.max()
    return np.exp(x) / np.exp(x).sum()

class Expert:
    def __init__(self, expert_id, d_in, d_out):
        self.id = expert_id
        self.W = np.random.randn(d_in, d_out) * 0.1
        self.specialization = f'Expert_{expert_id}'

    def forward(self, x):
        return np.tanh(x @ self.W)

class TopKGating:
    """Routes each token to top-k experts"""
    def __init__(self, d_model, n_experts):
        self.W_gate = np.random.randn(d_model, n_experts) * 0.1

    def forward(self, x, k=2):
        logits = x @ self.W_gate          # (n_experts,)
        # Add noise for load balancing
        noise = np.random.randn(len(logits)) * 0.01
        logits_noisy = logits + noise
        # Select top-k experts
        top_k_indices = np.argsort(logits_noisy)[-k:][::-1]
        top_k_logits  = logits[top_k_indices]
        weights = softmax(top_k_logits)   # routing weights sum to 1
        return top_k_indices, weights

class MixtureOfExperts:
    def __init__(self, n_experts, d_model, k=2):
        self.n_experts = n_experts
        self.k = k
        self.experts = [Expert(i, d_model, d_model) for i in range(n_experts)]
        self.gating   = TopKGating(d_model, n_experts)
        self.expert_load = np.zeros(n_experts)

    def forward(self, x):
        expert_indices, weights = self.gating.forward(x, k=self.k)
        # Track which experts are used
        for idx in expert_indices:
            self.expert_load[idx] += 1
        # Combine expert outputs
        output = np.zeros_like(x)
        for expert_idx, weight in zip(expert_indices, weights):
            expert_out = self.experts[expert_idx].forward(x)
            output += weight * expert_out
        return output, expert_indices, weights

np.random.seed(42)
d_model, n_experts, k = 8, 4, 2
moe = MixtureOfExperts(n_experts=n_experts, d_model=d_model, k=k)

print(f'MoE: {n_experts} experts, routing top-{k} per token')
print('=== Processing 5 tokens ===')

for t in range(5):
    token = np.random.randn(d_model)
    output, selected, weights = moe.forward(token)
    print(f'  Token {t+1}: routed to experts {selected} with weights {weights.round(3)}')

print(f'\n=== Expert Load Distribution ===')
for i, load in enumerate(moe.expert_load):
    bar = '█' * int(load)
    print(f'  Expert {i}: {int(load)} tokens {bar}')
print(f'\n✅ Only {k}/{n_experts} experts activated per token — {k/n_experts:.0%} compute used!')

---
# 🔷 BONUS: Key Concepts Quick Reference
---

In [ ]:
# BONUS: Quick reference implementations of important components
import numpy as np

# === Layer Normalization (used in Transformers) ===
def layer_norm(x, gamma=None, beta=None, eps=1e-6):
    """Normalize across feature dimension — stabilizes training"""
    mean = x.mean(axis=-1, keepdims=True)
    var  = x.var(axis=-1, keepdims=True)
    x_norm = (x - mean) / np.sqrt(var + eps)
    if gamma is not None:
        x_norm = gamma * x_norm + beta
    return x_norm

# === Feed-Forward Network (used in each Transformer layer) ===
def feed_forward(x, W1, b1, W2, b2):
    """FFN(x) = max(0, xW1+b1)W2+b2  (GELU/ReLU activation)"""
    hidden = np.maximum(0, x @ W1 + b1)  # ReLU
    return hidden @ W2 + b2

# === Token Generation (Autoregressive) ===
def greedy_decode(logits, vocab):
    """Pick most probable token at each step"""
    token_id = np.argmax(logits)
    return vocab[token_id]

def top_k_sampling(logits, k=5, temperature=1.0, vocab=None):
    """Sample from top-k tokens with temperature scaling"""
    scaled = logits / temperature
    top_k_indices = np.argsort(scaled)[-k:]
    top_k_logits  = scaled[top_k_indices]
    probs = np.exp(top_k_logits - top_k_logits.max())
    probs /= probs.sum()
    chosen = np.random.choice(top_k_indices, p=probs)
    return vocab[chosen] if vocab else chosen

def nucleus_sampling(logits, p=0.9, temperature=1.0, vocab=None):
    """Sample from smallest set of tokens whose cumulative probability >= p"""
    scaled = logits / temperature
    probs  = np.exp(scaled - scaled.max())
    probs /= probs.sum()
    sorted_idx  = np.argsort(probs)[::-1]
    sorted_probs = probs[sorted_idx]
    cumsum = np.cumsum(sorted_probs)
    nucleus = sorted_idx[cumsum <= p]
    if len(nucleus) == 0:
        nucleus = sorted_idx[:1]
    nucleus_probs = probs[nucleus]
    nucleus_probs /= nucleus_probs.sum()
    chosen = np.random.choice(nucleus, p=nucleus_probs)
    return vocab[chosen] if vocab else chosen

# Demo
np.random.seed(42)
vocab = ['the', 'cat', 'sat', 'on', 'mat', 'dog', 'ran', 'fast', 'slow', 'big']
logits = np.array([2.1, 1.5, 0.8, 0.3, -0.5, 1.8, 0.6, -0.2, 0.1, 0.4])

print('=== Token Generation Strategies ===')
print(f'Logits: {dict(zip(vocab, logits.round(1)))}')
print(f'\nGreedy decode:      "{greedy_decode(logits, vocab)}" (always picks highest)')

print('\nTop-5 sampling (10 trials):')
samples = [top_k_sampling(logits, k=5, temperature=1.0, vocab=vocab) for _ in range(10)]
print(f'  {samples}')

print('\nNucleus (p=0.9) sampling (10 trials):')
samples_nuc = [nucleus_sampling(logits, p=0.9, temperature=1.0, vocab=vocab) for _ in range(10)]
print(f'  {samples_nuc}')

print('\n=== Temperature Effect ===')
for temp in [0.1, 0.5, 1.0, 2.0]:
    samples_t = [top_k_sampling(logits, k=5, temperature=temp, vocab=vocab) for _ in range(20)]
    unique = len(set(samples_t))
    print(f'  temp={temp}: diversity = {unique}/20 unique tokens | {samples_t[:5]}...')

---
# 📚 Summary: All 60 Questions Covered

| Section | Questions Covered |
|---|---|
| **Generative Models** | Q1–Q12: GAN, VAE, Mode Collapse, Overfitting, Gradient Clipping, Curriculum Learning, LR Scheduling, L1/L2 Loss, Gradient Penalty |
| **Large Language Models** | Q13–Q22: Transfer Learning, GPT vs BERT, RNN vs Transformer, Positional Encoding, Attention Mechanisms, Multi-Head Attention, RLHF, Catastrophic Forgetting |
| **Multimodal Models** | Q23–Q25: Cross-Modal Attention, Image Synthesis Losses, CLIP Contrastive Learning |
| **Embeddings** | Q26–Q29: Word Embeddings, Triplet Loss, Quantization, ANN Search |
| **Training & Evaluation** | Q30–Q32: BLEU/ROUGE/Perplexity, Hallucination Detection, Mixture of Experts |
| **Bonus** | Token Generation (Greedy, Top-K, Nucleus), Layer Norm, Feed-Forward |

---
> 💡 **Tips for Interview Prep**:
> - Understand the **math** behind each concept (loss functions, attention formulas)
> - Be able to explain **trade-offs** (L1 vs L2, GPT vs BERT)
> - Know **real-world applications** for each technique
> - Practice coding attention from scratch — it's a common whiteboard question